<a href="https://colab.research.google.com/github/ksenia-andreeva/kan-physics-recovery/blob/main/kan_vs_mlp_sp1_noise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практика: Восстановление физического закона из данных**

Задача: сгенерировать данные затухающего гармонического осциллятора и восстановить закон движения для разных значений гауссова шума.

# **Установка библиотек**

In [33]:
!pip install git+https://github.com/KindXiaoming/pykan.git

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.integrate import solve_ivp
from kan import KAN
from kan.utils import ex_round

torch.set_default_dtype(torch.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Работает на устройстве: {device}")

  Cloning https://github.com/KindXiaoming/pykan.git to /tmp/pip-req-build-r5_6_jej
  Running command git clone --filter=blob:none --quiet https://github.com/KindXiaoming/pykan.git /tmp/pip-req-build-r5_6_jej
  Resolved https://github.com/KindXiaoming/pykan.git to commit ecde4ec3274d3bef1ad737479cf126aed38ab530
  Preparing metadata (setup.py) ... done
Работает на устройстве: cpu


# **Генерация данных и обучение KAN и MLP для одного уровня шума**

In [34]:
def run_experiment(noise_level):
    # --- Генерация данных с шумом ---
    k, m, c = 4.0, 1.0, 0.3
    all_data = []

    def rhs(t, y):
        x, v = y
        a = -(k/m)*x - (c/m)*v
        return [v, a]

    for x0 in [1.0, 0.5, 2.0]:
        for v0 in [0.0, 0.5, -0.5]:
            sol = solve_ivp(rhs, (0, 20), y0=[x0, v0], t_eval=np.linspace(0, 20, 500))
            x, v = sol.y
            a_clean = -(k/m)*x - (c/m)*v
            a_std = np.std(a_clean)
            noise = np.random.normal(0, noise_level * a_std, size=len(a_clean))
            a_noisy = a_clean + noise

            for i in range(len(x)):
                all_data.append([x[i], v[i], a_noisy[i]])

    all_data = np.array(all_data)
    X = all_data[:, :2]
    y = all_data[:, 2]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    X_train_t = torch.tensor(X_train.astype(np.float32))
    y_train_t = torch.tensor(y_train.astype(np.float32)).reshape(-1, 1)
    X_test_t  = torch.tensor(X_test.astype(np.float32))
    y_test_t  = torch.tensor(y_test.astype(np.float32)).reshape(-1, 1)

    dataset = {
        'train_input': X_train_t,
        'train_label': y_train_t,
        'test_input': X_test_t,
        'test_label': y_test_t
    }

    # --- Обучение KAN ---
    model_kan = KAN(width=[2,2,1], grid=10, k=1, seed=42, device=device)
    model_kan.fit(dataset, opt="LBFGS", steps=800, lamb=0.0, lamb_entropy=0.0)
    model_kan = model_kan.prune()

    with torch.no_grad():
        pred_kan = model_kan(X_test_t)
        mse_kan = nn.functional.mse_loss(pred_kan, y_test_t).item()

    x_test = torch.tensor([[1.0, 0.0], [0.0, 1.0]], dtype=torch.float32)
    with torch.no_grad():
        a_pred = model_kan(x_test).numpy().flatten()
    alpha_kan, beta_kan = a_pred[0], a_pred[1]

    lib = ['x','x^2','x^3','x^4','exp','log','sqrt','tanh','sin','abs']
    model_kan.auto_symbolic(lib=lib)

    formula = ex_round(model_kan.symbolic_formula()[0][0],4)
    print("Символьная формула KAN:", formula)

    # --- Обучение MLP ---
    class MLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(2, 16),
                nn.ReLU(),
                nn.Linear(16, 1))
        def forward(self, x):
            return self.net(x)

    model_mlp = MLP()
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model_mlp.parameters(), lr=0.001)

    X_tr, y_tr = dataset['train_input'], dataset['train_label']
    X_te, y_te = dataset['test_input'], dataset['test_label']

    for epoch in range(6000):
        model_mlp.train()
        optimizer.zero_grad()
        loss = criterion(model_mlp(X_tr), y_tr)
        loss.backward()
        optimizer.step()

    model_mlp.eval()
    with torch.no_grad():
        mse_mlp = criterion(model_mlp(X_te), y_te).item()

    result = {
        'noise': noise_level,
        'mse_kan': mse_kan,
        'mse_mlp': mse_mlp,
        'formula': str(formula),
        'alpha_kan': alpha_kan,
        'beta_kan': beta_kan,
        'true_alpha': -k/m,
        'true_beta': -c/m
    }

    return result

# **Цикл по разному гауссовому шуму и сбор результатов**

In [35]:
noise_levels = [0.0, 0.01, 0.03, 0.05] # проверяемые уровни шума

results = []
for nl in noise_levels:
    print(f"Обработка шума {nl*100:.0f}%")
    res = run_experiment(nl)
    results.append(res)

Обработка шума 0%
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 1.42e-03 | test_loss: 1.57e-03 | reg: 1.74e+01 | : 100%|█| 800/800 [02:34<00:00,  5.18


saving model version 0.1
saving model version 0.2
fixing (0,0,0) with 0
fixing (0,0,1) with exp, r2=1.000000238418579, c=2
fixing (0,1,0) with 0
fixing (0,1,1) with exp, r2=1.0000003576278687, c=2
fixing (1,0,0) with 0
fixing (1,1,0) with exp, r2=1.0000003576278687, c=2
saving model version 0.3
Символьная формула KAN: 51.161 - 55.165*exp(0.0001*exp(4.0403*x_2) - 0.0754*exp(-10.0*x_1))
Обработка шума 1%
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 1.57e-02 | test_loss: 1.62e-02 | reg: 1.71e+01 | : 100%|█| 800/800 [01:53<00:00,  7.04


saving model version 0.1
saving model version 0.2
fixing (0,0,0) with exp, r2=1.000000238418579, c=2
fixing (0,1,0) with exp, r2=1.000000238418579, c=2
fixing (1,0,0) with exp, r2=1.000000238418579, c=2
saving model version 0.3
Символьная формула KAN: 22.2392 - 21.9612*exp(0.0132*exp(2.6*x_1) + 0.0004*exp(3.4*x_2))
Обработка шума 3%
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 7.46e-02 | test_loss: 7.59e-02 | reg: 1.69e+01 | : 100%|█| 800/800 [01:26<00:00,  9.22


saving model version 0.1
saving model version 0.2
fixing (0,0,0) with 0
fixing (0,0,1) with exp, r2=1.0000003576278687, c=2
fixing (0,1,0) with 0
fixing (0,1,1) with exp, r2=1.0000003576278687, c=2
fixing (1,0,0) with 0
fixing (1,1,0) with exp, r2=1.0000003576278687, c=2
saving model version 0.3
Символьная формула KAN: 29.0586 - 32.8915*exp(-0.1211*exp(-6.7516*x_1))
Обработка шума 5%
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 7.93e-02 | test_loss: 8.22e-02 | reg: 1.50e+01 | : 100%|█| 800/800 [02:58<00:00,  4.47


saving model version 0.1
saving model version 0.2
fixing (0,0,0) with x, r2=0.9999834299087524, c=1
fixing (0,0,1) with exp, r2=1.000000238418579, c=2
fixing (0,1,0) with exp, r2=1.000000238418579, c=2
fixing (0,1,1) with exp, r2=1.000000238418579, c=2
fixing (1,0,0) with exp, r2=1.0000003576278687, c=2
fixing (1,1,0) with exp, r2=1.0000003576278687, c=2
saving model version 0.3
Символьная формула KAN: -0.0059*exp(4.2897*x_1 - 1.4054*exp(-9.9995*x_2)) + 55.9776 - 59.9046*exp(-0.0669*exp(-10.0*x_1))


# **Вывод таблицы по уровням шума**

In [36]:
import pandas as pd

# сборка DataFrame
df = pd.DataFrame(results)
df['noise'] = df['noise'].apply(lambda x: f"{int(x*100)}%")

# Вывод таблицы с результатами
print("\n=== Таблица результатов ===")
print(df[[
    'noise',
    'mse_kan',
    'mse_mlp',
    'true_alpha',
    'alpha_kan',
    'true_beta',
    'beta_kan',
    'formula'
]].to_string(index=False))


=== Таблица результатов ===
noise  mse_kan      mse_mlp  true_alpha  alpha_kan  true_beta  beta_kan                                                                                           formula
   0% 0.000864 5.183458e-07        -4.0  -4.009368       -0.3 -0.293708                               51.161 - 55.165*exp(0.0001*exp(4.0403*x_2) - 0.0754*exp(-10.0*x_1))
   1% 0.000288 2.718270e-04        -4.0  -4.002959       -0.3 -0.298098                                  22.2392 - 21.9612*exp(0.0132*exp(2.6*x_1) + 0.0004*exp(3.4*x_2))
   3% 0.007764 2.411022e-03        -4.0  -3.829783       -0.3 -0.334910                                                   29.0586 - 32.8915*exp(-0.1211*exp(-6.7516*x_1))
   5% 0.006763 6.699655e-03        -4.0  -4.032134       -0.3 -0.280189 -0.0059*exp(4.2897*x_1 - 1.4054*exp(-9.9995*x_2)) + 55.9776 - 59.9046*exp(-0.0669*exp(-10.0*x_1))
